# Sepsis detection - Multivariate Time-Series Classification problem

In [1]:
#! python3 -m pip install pandas seaborn numpy matplotlib scikit-learn gboost scipy "spacy==3.7.5" imbalanced-learn tensorflow

## Importing libraries

In [2]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

## Load the dataset and inspects few infromations of the dataset

In [3]:
dataset = pd.read_csv("./Dataset.csv") # load dataset

In [4]:
dataset.head(5) # view some rows

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [5]:
dataset.columns # dataset columns

Index(['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP',
       'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
       'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
       'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
       'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
       'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
       'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID'],
      dtype='object')

## Preprocessing the data : split first, input and scale, add padding and drop

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

### Split first (7/1/2) then defining a preprocessor to preprocess data


For numerical features, it will be impute using constant value to avoid leakage and then scale using standard scaler

For categorical features, it will also impute using constant and label encoded it

In [7]:
# Split train validation test FIRST before touching the data
patient_ids_before_split = []
patient_labels_before_split = []
groups_before_split = dataset.groupby("Patient_ID")

for patient_id, patient_df in groups_before_split:
    label = 0
    for i in patient_df["SepsisLabel"]:
        if i == 1:
            label = 1

    patient_ids_before_split.append(patient_id)
    patient_labels_before_split.append(label)

patient_labels = pd.DataFrame({
    "Patient_ID": patient_ids_before_split,
    "SepsisLabel": patient_labels_before_split
})

In [8]:
train_patients, other_patients = train_test_split(
    patient_labels, test_size= 0.3, random_state= 67, stratify=patient_labels["SepsisLabel"]
) # 70 train and leave 30 for validation + testing

validation_patients, test_patients = train_test_split(
    other_patients, test_size= 2/3, random_state= 67, stratify=other_patients["SepsisLabel"]
) # from the 30: 10 validation and 20 testing


dataset_train = dataset[dataset["Patient_ID"].isin(train_patients["Patient_ID"])].copy()
dataset_validation = dataset[dataset["Patient_ID"].isin(validation_patients["Patient_ID"])].copy()
dataset_test = dataset[dataset["Patient_ID"].isin(test_patients["Patient_ID"])].copy()

print(len(train_patients), len(validation_patients), len(test_patients))
print(train_patients["SepsisLabel"].value_counts(normalize=True).sort_index())
print(validation_patients["SepsisLabel"].value_counts(normalize=True).sort_index())
print(test_patients["SepsisLabel"].value_counts(normalize=True).sort_index())



28235 4033 8068
SepsisLabel
0    0.927324
1    0.072676
Name: proportion, dtype: float64
SepsisLabel
0    0.927349
1    0.072651
Name: proportion, dtype: float64
SepsisLabel
0    0.927243
1    0.072757
Name: proportion, dtype: float64


## ---------------------------------------------------------------------------------------------------------------------------------------------------------

## Gradient Boosting with lag features — Raden

In [9]:

vital_cols = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]

def add_lag_features(df):
    df = df.sort_values(["Patient_ID", "Hour"]).copy()
    patients = df.groupby("Patient_ID")

    hour_gap = patients["Hour"].diff().dropna()
    assert hour_gap.eq(1).all(), "Patient records contain hourly gaps"

    for lag in [1, 3, 6]:
        previous = patients[vital_cols].shift(lag)

        for col in vital_cols:
            df[f"{col}_lag_{lag}h"] = previous[col]

  
    for col in vital_cols:
        df[f"{col}_change_1h"] = df[col] - df[f"{col}_lag_1h"]

    return df


gb_train_raw = add_lag_features(dataset_train)
gb_validation_raw = add_lag_features(dataset_validation)
gb_test_raw = add_lag_features(dataset_test)

gb_train_raw[
    ["Patient_ID", "Hour", "HR",
     "HR_lag_1h", "HR_lag_3h", "HR_lag_6h", "HR_change_1h"]
].head(10)

,Patient_ID,Hour,HR,HR_lag_1h,HR_lag_3h,HR_lag_6h,HR_change_1h
132940,1,0,NaN,NaN,NaN,NaN,NaN
132941,1,1,97.0,NaN,NaN,NaN,NaN
132942,1,2,89.0,97.0,NaN,NaN,-8.0
132943,1,3,90.0,89.0,NaN,NaN,1.0
132944,1,4,103.0,90.0,97.0,NaN,13.0
132945,1,5,110.0,103.0,89.0,NaN,7.0
132946,1,6,108.0,110.0,90.0,NaN,-2.0
132947,1,7,106.0,108.0,103.0,97.0,-2.0
132948,1,8,104.0,106.0,110.0,89.0,-2.0
132949,1,9,102.0,104.0,108.0,90.0,-2.0


In [10]:
gb_window = int(np.percentile(
    dataset_train.groupby("Patient_ID").size(), 95
))


gb_features = [
    col for col in gb_train_raw.columns
    if col not in ["Patient_ID", "SepsisLabel", "Unnamed: 0"]
]

def prepare_gb_patients(df):
    rows, labels = [], []

    for patient_id, patient in df.groupby("Patient_ID"):
        patient = patient.sort_values("Hour")

        if len(patient) > gb_window:
            continue

        values = patient[gb_features].to_numpy(dtype=np.float32)

        padded = np.full(
            (gb_window, len(gb_features)),
            np.nan,
            dtype=np.float32
        )
        padded[-len(values):] = values

        rows.append(padded.reshape(-1))
        labels.append(int(patient["SepsisLabel"].max()))

    return np.stack(rows), np.asarray(labels)

X_gb_train, y_gb_train = prepare_gb_patients(gb_train_raw)
X_gb_val, y_gb_val = prepare_gb_patients(gb_validation_raw)
X_gb_test, y_gb_test = prepare_gb_patients(gb_test_raw)

print("Training:", X_gb_train.shape)
print("Validation:", X_gb_val.shape)
print("Test:", X_gb_test.shape)

Training: (27017, 4002)
Validation: (3867, 4002)
Test: (7733, 4002)


In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

positive_weight = (
    (y_gb_train == 0).sum() / (y_gb_train == 1).sum()
)

gb_lag_model = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        keep_empty_features=True
    )),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=67,
        verbose=1
    ))
])

sample_weights = np.where(
    y_gb_train == 1, positive_weight, 1.0
)

gb_lag_model.fit(
    X_gb_train,
    y_gb_train,
    model__sample_weight=sample_weights
)

gb_val_probability = gb_lag_model.predict_proba(
    X_gb_val
)[:, 1]

print("Gradient Boosting + raw lag features — validation")

print(classification_report(
    y_gb_val,
    (gb_val_probability >= 0.30).astype(int),
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(
    y_gb_val, gb_val_probability
))

      Iter       Train Loss   Remaining Time 
         1           1.3369            7.10m
         2           1.2948            7.00m
         3           1.2599            6.90m
         4           1.2270            6.81m
         5           1.1994            6.75m
         6           1.1724            6.66m
         7           1.1503            6.60m
         8           1.1286            6.51m
         9           1.1104            6.45m
        10           1.0930            6.37m
        20           0.9686            5.69m
        30           0.8889            5.00m
        40           0.8286            4.67m
        50           0.7804            4.13m
        60           0.7405            3.43m
        70           0.7056            2.63m
        80           0.6762            1.80m
        90           0.6502           54.59s
       100           0.6267            0.00s
Gradient Boosting + raw lag features — validation
              precision    recall  f1-score   sup

In [12]:
# Final test evaluation Threshold at 0.5 or 0.3 (?)
gb_threshold = 0.30

gb_test_probability = gb_lag_model.predict_proba(X_gb_test)[:, 1]
gb_test_prediction = (
    gb_test_probability >= gb_threshold
).astype(int)

print("gboost + lag features — TEST results")
print(classification_report(
    y_gb_test,
    gb_test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(y_gb_test, gb_test_probability))

gboost + lag features — TEST results
              precision    recall  f1-score   support

   No sepsis     0.9902    0.7414    0.8479      7347
      Sepsis     0.1487    0.8601    0.2536       386

    accuracy                         0.7473      7733
   macro avg     0.5695    0.8007    0.5508      7733
weighted avg     0.9482    0.7473    0.8182      7733

ROC-AUC: 0.9003431663976202


## ---------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
# transforming pipeline for numerical features
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value= 0)), ("scaler", StandardScaler())]
)

# transforming pipeline for catergorical features
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value = "UNKNOWN")), # no data leak hehe
        ("encoder", LabelEncoder()),
    ]
)

# Extract out catergorical and numerical columns
cat_cols = []
num_cols = []
for col_name, dtype in dataset_train.dtypes.items():

    if col_name == "SepsisLabel" or col_name == "Patient_ID":
        continue  # SKIP the label and id of the patient

    if np.issubdtype(dtype, np.number):
        num_cols.append(col_name)
    else:
        cat_cols.append(col_name)


# define precessors
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols,
        ),
        
    ],
    remainder="passthrough" # for the label and id
)

columns = dataset.columns

data_train_processed = preprocessor.fit_transform(dataset_train) # ONLY fit using train because no leak
data_validation_processed = preprocessor.transform(dataset_validation)
data_test_processed = preprocessor.transform(dataset_test)

dataset_train_reconstruct = pd.DataFrame(data_train_processed, columns= columns) # reconstruct back to time-series preocessing
dataset_validation_reconstruct = pd.DataFrame(data_validation_processed, columns= columns)
dataset_test_reconstruct = pd.DataFrame(data_test_processed, columns= columns)

In [14]:
print(dataset_train_reconstruct.shape) # correct dimensions
print(dataset_validation_reconstruct.shape)
print(dataset_test_reconstruct.shape)

(1088003, 44)
(154718, 44)
(309489, 44)


### Time-series processing: 

1. Find 95% of TRAIN patients window lenght (How many hours 95% percentile patients data last)

2. Drop all patients above that. For patients who above 95%, a sliding window technique can be apply where it is expensive but less expensive to train (trade-off).

3. Transform the problem to detection problem. As a patient can have Sepsis in multiple hours but not others. If there is sepsis in any hour, the patient is classified as sepsis

NOTE: justification to remain time-series problem rather time-step prediction ( simple supervised problem ): temporal information can carries informations about the patient information hence having full window retains useful information

In [15]:
groups_train = dataset_train_reconstruct.groupby("Patient_ID") # Group patients records
groups_validation = dataset_validation_reconstruct.groupby("Patient_ID")
groups_test = dataset_test_reconstruct.groupby("Patient_ID")

hours =[] # how many hours is each patient records last

# Analysis of the patient time lentgh
for patient_id, patient_df in groups_train:
    hours.append(patient_df.shape[0])

window = np.percentile(hours,95) # 95% percentile ONLY from train data
window # 58 hours


np.float64(58.0)

In [16]:
# preprocessing : drop all > 95% percentile hours and add padding
def add_padding(groups,window):

    target_var = []
    transformed_sequence = []


    for patient_id, patient_df in groups:

        #extract the target variable
        # if the patient have sepsis, they will have at least 1 row to be 1
        label = 0
        for i in patient_df["SepsisLabel"]:
            if i == 1:
                label = 1

        patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])

        if patient_df.shape[0] > window:
            #im lazy sorry:(
            continue

        target_var.append(label)
        #add padding
        patient_df = patient_df.sort_values("Hour")
        
        num_feature = patient_df.shape[1] # should be 43 anyway


        #change to np
        patient_np = patient_df.to_numpy()
        num_padding_rows = int(window) - patient_df.shape[0]
        all_zeros = np.zeros(num_feature)
        new_patient = []
        for i in range (num_padding_rows):
            new_patient.append(all_zeros)
        for hour in patient_np:
            new_patient.append(hour)

        transformed_sequence.append(new_patient)

    transformed_sequence = np.asarray(transformed_sequence)
    target_var = np.asarray(target_var)
    return transformed_sequence,target_var

# Add padding to all set: train , validation, test
X_train,y_train = add_padding(groups_train,window)
X_validation,y_validation = add_padding(groups_validation,window)
X_test,y_test = add_padding(groups_test,window)

print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)

(27017, 58, 42)
(3867, 58, 42)
(7733, 58, 42)


In [17]:
X_train[0:5] # Expect a 3D array 

array([[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 8.71051109e-01,  8.71051109e-01,  2.93879153e-01, ...,
         -6.58013630e-01,  3.38370073e-01,  8.48199048e-01],
        [ 9.05308716e-01,  9.05308716e-01,  3.27149536e-01, ...,
         -6.58013630e-01,  3.38370073e-01,  8.82250589e-01],
        [ 9.39566324e-01,  9.39566324e-01,  2.60608769e-01, ...,
         -6.58013630e-01,  3.38370073e-01,  9.16302130e-01]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [18]:
y_train # this is our ground truth for train

array([0, 0, 0, ..., 0, 0, 0], shape=(27017,))

In [19]:
target_var_train_np = np.asarray(y_train) # transform to np array
target_var_validation_np = np.asarray(y_validation)
target_var_test_np = np.asarray(y_test)

In [20]:
print(target_var_train_np.shape) # check shape
print(target_var_validation_np.shape)
print(target_var_test_np.shape)

(27017,)
(3867,)
(7733,)


In [21]:
# Already split 7/1/2 before preprocessing but check for sure
print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)
print(len(y_train))
print(len(y_validation))
print(len(y_test))

(27017, 58, 42)
(3867, 58, 42)
(7733, 58, 42)
27017
3867
7733


In [22]:
print(len([one for one in y_train if one == 0])) # How non-sepsis sepsis is distributed (Data is VERY IMBALANCE)

25696


### Upsampling by SMOTE
1. Flatten out to SMOTE

2. Apply SMOTE (SMOTE perturbs and use KKN to get the neighbours). This upsampling under-represented label

3. Fit SMOTE and yeah! ONLY TRAIN THOUGH validation and testing remains real

In [23]:
#SMOTE sampling X_train

#Flatten to 2D

num_sample, num_time, num_feature = X_train.shape

X_train_flatten = X_train.reshape(num_sample, num_time*num_feature) #Flatten to do SMOTE
#SMOTE time
s = SMOTE(k_neighbors=10, random_state= 67)
X_train_sampled_flatten , y_train_sampled = s.fit_resample(X_train_flatten,y_train)
X_train_sampled = X_train_sampled_flatten.reshape(-1, num_time, num_feature) # reshape back to 3D

X_train_sampled.shape

(51392, 58, 42)

In [24]:
# check shapes
print(len(y_train_sampled))

print(len([one for one in y_train_sampled if one == 1]))
print(len([one for one in y_train_sampled if one == 0]))

# data is balanced now

51392
25696
25696


In [25]:
# recontruct back to np array
X_train_sampled = np.asarray(X_train_sampled, dtype=np.float32)
y_train_sampled = np.asarray(y_train_sampled, dtype=np.float32)
X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
X_validation = np.asarray(X_validation, dtype=np.float32)
y_validation = np.asarray(y_validation, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

## ---------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

X_sampled_flat = X_train_sampled.reshape(
    X_train_sampled.shape[0], -1
)

X_validation_flat = X_validation.reshape(
    X_validation.shape[0], -1
)

gb_shared = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=67,
    verbose=1
)

gb_shared.fit(X_sampled_flat, y_train_sampled)

shared_val_probability = gb_shared.predict_proba(
    X_validation_flat
)[:, 1]

print("Gradient Boosting using shared SMOTE data — validation")

print(classification_report(
    y_validation,
    (shared_val_probability >= 0.30).astype(int),
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(
    y_validation, shared_val_probability
))

      Iter       Train Loss   Remaining Time 
         1           1.2369           20.66m
         2           1.1109           20.16m
         3           1.0060           20.09m
         4           0.9185           19.94m
         5           0.8426           19.20m
         6           0.7769           18.07m
         7           0.7202           17.70m
         8           0.6702           17.38m
         9           0.6247           17.41m
        10           0.5867           17.28m
        20           0.3685           14.18m
        30           0.2837           11.54m
        40           0.2379            9.53m
        50           0.2066            7.78m
        60           0.1878            6.20m
        70           0.1711            4.66m
        80           0.1617            3.08m
        90           0.1538            1.53m
       100           0.1470            0.00s
Gradient Boosting using shared SMOTE data — validation
              precision    recall  f1-score 

In [27]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

lag_columns = [
    col for col in gb_train_raw.columns
    if "_lag_" in col or "_change_" in col
]

lag_preprocessor = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant", fill_value=0,
        keep_empty_features=True
    )),
    ("scaler", StandardScaler())
])

lag_preprocessor.fit(gb_train_raw[lag_columns])

def prepare_processed_lags(df):
    df = df.sort_values(["Patient_ID", "Hour"]).copy()

    df[lag_columns] = lag_preprocessor.transform(df[lag_columns])

    sequences, labels = [], []

    for patient_id, patient in df.groupby("Patient_ID"):
        if len(patient) > window:
            continue

        values = patient[lag_columns].to_numpy(dtype=np.float32)

        padded = np.zeros(
            (int(window), len(lag_columns)), dtype=np.float32
        )
        padded[-len(values):] = values

        sequences.append(padded)
        labels.append(int(patient["SepsisLabel"].max()))

    return np.stack(sequences), np.asarray(labels)

train_lags, train_lag_labels = prepare_processed_lags(gb_train_raw)
val_lags, val_lag_labels = prepare_processed_lags(gb_validation_raw)
test_lags, test_lag_labels = prepare_processed_lags(gb_test_raw)

assert np.array_equal(train_lag_labels, y_train)
assert np.array_equal(val_lag_labels, y_validation)
assert np.array_equal(test_lag_labels, y_test)

X_train_lag = np.concatenate([X_train, train_lags], axis=2)
X_validation_lag = np.concatenate([X_validation, val_lags], axis=2)
X_test_lag = np.concatenate([X_test, test_lags], axis=2)

X_train_lag_flat = X_train_lag.reshape(len(X_train_lag), -1)
X_validation_lag_flat = X_validation_lag.reshape(len(X_validation_lag), -1)
X_test_lag_flat = X_test_lag.reshape(len(X_test_lag), -1)

X_train_lag_sampled, y_train_lag_sampled = SMOTE(
    k_neighbors=10, random_state=67
).fit_resample(X_train_lag_flat, y_train)

print("Expanded sampled training data:", X_train_lag_sampled.shape)

Expanded sampled training data: (51392, 4060)


In [28]:
from sklearn.base import clone

gb_shared_lags = clone(gb_shared)

gb_shared_lags.fit(
    X_train_lag_sampled,
    y_train_lag_sampled
)

shared_lag_val_probability = gb_shared_lags.predict_proba(
    X_validation_lag_flat
)[:, 1]

print("Gradient Boosting + lags using shared SMOTE data — validation")

print(classification_report(
    y_validation,
    (shared_lag_val_probability >= 0.30).astype(int),
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0
))

print("ROC-AUC:", roc_auc_score(
    y_validation, shared_lag_val_probability
))

      Iter       Train Loss   Remaining Time 
         1           1.2501           36.64m
         2           1.1368           35.96m
         3           1.0418           35.51m
         4           0.9616           35.09m
         5           0.8921           34.79m
         6           0.8319           34.47m
         7           0.7799           34.15m
         8           0.7340           33.88m
         9           0.6932           33.53m
        10           0.6570           33.42m
        20           0.4405           29.45m
        30           0.3372           25.92m
        40           0.2746           22.23m
        50           0.2392           18.56m
        60           0.2148           14.86m
        70           0.1946           11.15m
        80           0.1793            7.44m
        90           0.1676            3.72m
       100           0.1596            0.00s
Gradient Boosting + lags using shared SMOTE data — validation
              precision    recall  f1

## ---------------------------------------------------------------------------------------------------------------------------------------------------------

### Baseline: RNN (simple small) - Note: No pass this baseline on Recall Sepsis -> Try again

In [29]:
# Baseline SimpleRNN for the patient sequences
import random
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

num_time_steps, num_features = X_train_sampled.shape[1], X_train_sampled.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train_sampled,
    y_train_sampled,
    validation_data=(X_validation, y_validation),
    epochs=10,
    batch_size=100,
    verbose=2,
)

test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

c:\Users\jovan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
514/514 - 4s - 8ms/step - accuracy: 0.8683 - loss: 0.3129 - val_accuracy: 0.8986 - val_loss: 0.2492
Epoch 2/10
514/514 - 2s - 5ms/step - accuracy: 0.9189 - loss: 0.2110 - val_accuracy: 0.9237 - val_loss: 0.1893
Epoch 3/10
514/514 - 2s - 4ms/step - accuracy: 0.9338 - loss: 0.1741 - val_accuracy: 0.9247 - val_loss: 0.1897
Epoch 4/10
514/514 - 2s - 4ms/step - accuracy: 0.9405 - loss: 0.1571 - val_accuracy: 0.9227 - val_loss: 0.2061
Epoch 5/10
514/514 - 2s - 4ms/step - accuracy: 0.9457 - loss: 0.1441 - val_accuracy: 0.9240 - val_loss: 0.2054
Epoch 6/10
514/514 - 2s - 4ms/step - accuracy: 0.9500 - loss: 0.1307 - val_accuracy: 0.9273 - val_loss: 0.1969
Epoch 7/10
514/514 - 2s - 5ms/step - accuracy: 0.9542 - loss: 0.1236 - val_accuracy: 0.9160 - val_loss: 0.2459
Epoch 8/10
514/514 - 2s - 5ms/step - accuracy: 0.9561 - loss: 0.1191 - val_accuracy: 0.9434 - val_loss: 0.1622
Epoch 9/10
514/514 - 2s - 5ms/step - accuracy: 0.9589 - loss: 0.1127 - val_accuracy: 0.9263 - val_loss: 0.2177
E

In [30]:

num_time_steps, num_features = X_train.shape[1], X_train.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=10,
    batch_size=100,
    verbose=2,
)

test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

c:\Users\jovan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
271/271 - 2s - 8ms/step - accuracy: 0.9131 - loss: 0.2130 - val_accuracy: 0.9578 - val_loss: 0.1324
Epoch 2/10
271/271 - 1s - 5ms/step - accuracy: 0.9604 - loss: 0.1257 - val_accuracy: 0.9589 - val_loss: 0.1251
Epoch 3/10
271/271 - 1s - 5ms/step - accuracy: 0.9624 - loss: 0.1189 - val_accuracy: 0.9599 - val_loss: 0.1224
Epoch 4/10
271/271 - 2s - 6ms/step - accuracy: 0.9637 - loss: 0.1147 - val_accuracy: 0.9599 - val_loss: 0.1212
Epoch 5/10
271/271 - 1s - 6ms/step - accuracy: 0.9648 - loss: 0.1129 - val_accuracy: 0.9607 - val_loss: 0.1221
Epoch 6/10
271/271 - 1s - 5ms/step - accuracy: 0.9649 - loss: 0.1105 - val_accuracy: 0.9604 - val_loss: 0.1209
Epoch 7/10
271/271 - 2s - 6ms/step - accuracy: 0.9664 - loss: 0.1069 - val_accuracy: 0.9610 - val_loss: 0.1194
Epoch 8/10
271/271 - 1s - 5ms/step - accuracy: 0.9671 - loss: 0.1045 - val_accuracy: 0.9610 - val_loss: 0.1190
Epoch 9/10
271/271 - 1s - 4ms/step - accuracy: 0.9683 - loss: 0.1023 - val_accuracy: 0.9604 - val_loss: 0.1185
E

In [31]:
import joblib
model.save("./baseline_rnn.keras")
joblib.dump(preprocessor, "./preprocessor")

['./preprocessor']

In [32]:
# Load the saved model and preprocessing pipeline
import joblib
import tensorflow as tf

model = tf.keras.models.load_model("./baseline_rnn.keras")
preprocessor = joblib.load("./preprocessor")

In [33]:
dataset_reconstruct

NameError: name 'dataset_reconstruct' is not defined

In [ ]:
dummy_num_rows = 10
# The saved preprocessor was fitted with all original columns, so use zero placeholders
# for the excluded columns only while transforming, then keep them for add_padding.
dummy_raw = pd.DataFrame(
    np.zeros((dummy_num_rows, len(dataset.columns))),
    columns=dataset.columns,
)
dummy_data = pd.DataFrame(
    preprocessor.transform(dummy_raw),
    columns=dataset.columns,
)

print(f"Dummy patient shape after preprocessing: {dummy_data.shape}")

def add_padding(patient_df, window):
    # Assumes that the patient has fewer rows than the window.
    for i in patient_df["SepsisLabel"]:
        print(i)
    patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])
    patient_df = patient_df.sort_values("Hour")

    num_feature = patient_df.shape[1]
    patient_np = patient_df.to_numpy()
    num_padding_rows = int(window) - patient_df.shape[0]
    all_zeros = np.zeros(num_feature)
    new_patient = []
    count = 0
    for i in range(num_padding_rows):
        new_patient.append(all_zeros)
        count += 1
    for hour in patient_np:
        new_patient.append(hour)
        count += 1
    new_patient_np = np.asarray(new_patient)
    print(count)
    print(len(new_patient))
    return new_patient_np

X_infer = add_padding(dummy_data, 58)[None, ...]
print(X_infer.shape)
y = model.predict(X_infer)
if y[0][0] > 0.5:
    print("Sepsis")

if y[0][0] < 0.5:
    print("No Sepsis")
print(y)

Dummy patient shape after preprocessing: (10, 44)
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
58
58
(1, 58, 42)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
No Sepsis
[[0.01691194]]
